# Phase 6 — Does splitting the action into two questions help?

### The hypothesis

Your baseline's confusion matrix showed something specific: when it misses a diagonal, ~80% of the time it picks **one of the diagonal's two halves** (UPLEFT → UP or LEFT). It understands direction. It just can't commit to "both."

The **factorized** model asks two 3-way questions instead of one 9-way question:

- **horizontal head:** LEFT / NONE / RIGHT
- **vertical head:** UP / NONE / DOWN

Its prediction is the pair: (LEFT, UP) = UPLEFT, (NONE, NONE) = NOOP, and so on. The idea is that diagonals stop being 4 rare classes with ~1% of the data each. Every UP clip also teaches the vertical head about UPLEFT, and every LEFT clip teaches the horizontal head about it.

### What makes this a fair test

**Only the head and the loss change.** Same backbone, same clips, same split, same seed, same 15 epochs, same learning rate, same training loop (`src/train.py`). The original repo also changed the loss weighting at the same time, so it couldn't tell what caused what. Here, any difference comes from factorizing.

The loss is simply the two questions' losses added:

`loss = CrossEntropy(horizontal guess, true horizontal) + CrossEntropy(vertical guess, true vertical)`

### Possible outcomes, and what each would mean

| Result | Interpretation |
| --- | --- |
| Diagonal recall goes up clearly, cardinals stay ~same | Factorizing helps rare combinations: a real finding |
| Diagonals no better | The diagonal problem isn't the head's fault. It supports the "diagonal presses are often invisible" idea from the baseline |
| Everything slightly worse | Splitting the question costs something (errors on two axes compound) |

All three are publishable. Don't root for one.

**Setup:** the same Drive folder as Phase 4. Upload the new **`src/train.py`** into `pacman-idm/src/` first. Runtime → **T4 GPU**.

## 1. Drive, files, local copy

In [ ]:
import os, sys, json, time, shutil

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/pacman-idm"
else:
    PROJECT = os.path.abspath(os.getcwd())
    while not os.path.exists(os.path.join(PROJECT, "README.md")) and os.path.dirname(PROJECT) != PROJECT:
        PROJECT = os.path.dirname(PROJECT)
sys.path.insert(0, PROJECT)

DRIVE_DATA = os.path.join(PROJECT, "data", "processed")
BASELINE_DIR = os.path.join(PROJECT, "checkpoints", "baseline_9way")
for f in ["src/labels.py", "src/data.py", "src/model.py", "src/train.py", "data/processed/frames.npy",
          "data/processed/splits.npz", "data/processed/meta.json", "checkpoints/baseline_9way/best.pth"]:
    print(("OK      " if os.path.exists(os.path.join(PROJECT, f)) else "MISSING ") + f)

if IN_COLAB:
    DATA = "/content/processed"
    os.makedirs(DATA, exist_ok=True)
    for f in ["frames.npy", "splits.npz", "meta.json"]:
        if not os.path.exists(os.path.join(DATA, f)):
            t = time.time(); shutil.copy(os.path.join(DRIVE_DATA, f), DATA)
            print(f"copied {f} in {time.time() - t:.0f}s")
else:
    DATA = DRIVE_DATA

## 2. Settings (identical to the baseline)

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

from src.data import ClipDataset
from src.model import IDM9Way, IDMFactorized, count_params
from src.train import ce_9way, ce_factorized, scores, evaluate, train

QUICK = True          # True first (a few minutes) to check it runs, then False for the real run
EPOCHS = 15
BATCH = 32
LR = 1e-4
WEIGHT_DECAY = 1e-5
SEED = 42             # same as the baseline
RUN_NAME = f"factorized_seed{SEED}" + ("_quick" if QUICK else "")

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = dev.type == "cuda"
print("device:", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "no GPU")
CKPT_DIR = os.path.join(PROJECT, "checkpoints", RUN_NAME)
torch.manual_seed(SEED); np.random.seed(SEED)

## 3. Data (same clips, same split)

In [ ]:
train_ds, val_ds, test_ds = (ClipDataset(DATA, s) for s in ["train", "val", "test"])
NAMES = train_ds.class_names
if QUICK:
    EPOCHS = 1
    g = np.random.default_rng(0)
    train_ds = Subset(train_ds, g.choice(len(train_ds), min(2000, len(train_ds)), replace=False))
    val_ds = Subset(val_ds, g.choice(len(val_ds), min(1000, len(val_ds)), replace=False))

workers = 2 if IN_COLAB else 0
train_loader = DataLoader(train_ds, BATCH, shuffle=True, num_workers=workers, pin_memory=USE_AMP, drop_last=True)
val_loader = DataLoader(val_ds, 128, num_workers=workers, pin_memory=USE_AMP)
test_loader = DataLoader(test_ds, 128, num_workers=workers, pin_memory=USE_AMP)
print(f"train {len(train_ds):,} | val {len(val_ds):,} | test {len(test_ds):,}")

## 4. The factorized model

Same backbone. The single `512 → 128 → 9` head becomes two `512 → 128 → 3` heads. That's 65,277 extra weights (+0.8%), so the comparison isn't about model size.

In [ ]:
model = IDMFactorized().to(dev)
print(f"factorized: {count_params(model):,} params | baseline: {count_params(IDM9Way()):,}")

xb, yb = next(iter(test_loader))
model.eval()
with torch.no_grad():
    h_logits, v_logits = model(xb[:4].to(dev))
print("horizontal head:", tuple(h_logits.shape), "| vertical head:", tuple(v_logits.shape))
print("true labels      :", [NAMES[i] for i in yb[:4]])
print("as (h, v) targets:", [(int(y % 3), int(y // 3)) for y in yb[:4]], " (h: 0=L 1=none 2=R, v: 0=U 1=none 2=D)")

## 5. Train

The same loop as the baseline, now from `src/train.py`. Each epoch line also shows **H** and **V**: how often the horizontal and vertical parts of the guess are right. Saved to Drive every epoch; if Colab disconnects, Run all again to resume.

In [ ]:
torch.manual_seed(SEED)
model = IDMFactorized().to(dev)
history, best_f1, best_path = train(model, ce_factorized, train_loader, val_loader, CKPT_DIR,
                                    EPOCHS, LR, WEIGHT_DECAY, dev, USE_AMP)

## 6. Test both models the same way

We reload your **baseline** checkpoint and score both models on test with the same code, so every number below is directly comparable.

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=dev))
_, fp, ty = evaluate(model, test_loader, ce_factorized, dev, USE_AMP)
fact = scores(fp, ty)

base_path = os.path.join(BASELINE_DIR, "best.pth")
have_base = os.path.exists(base_path)
if have_base:
    base_model = IDM9Way().to(dev)
    base_model.load_state_dict(torch.load(base_path, map_location=dev))
    _, bp, _ = evaluate(base_model, test_loader, ce_9way, dev, USE_AMP)
    base = scores(bp, ty)
else:
    print("No baseline checkpoint found, so only the factorized model is shown.")

rows = [("baseline 9-way", base)] if have_base else []
rows.append(("factorized", fact))
print(f"{'TEST':16s} {'acc':>7s} {'macro-F1':>9s} {'bal-acc':>8s} {'H axis':>7s} {'V axis':>7s}")
for name, s in rows:
    print(f"{name:16s} {s['acc']:7.3f} {s['macro_f1']:9.3f} {s['bal_acc']:8.3f} {s['h_acc']:7.3f} {s['v_acc']:7.3f}")

## 7. Per-action recall, side by side

Recall = of all real clips of that action, how many the model got right. **The diagonals are where the hypothesis lives.**

In [ ]:
def recall_per_class(p, y):
    cm = confusion_matrix(y, p, labels=range(9))
    return cm.diagonal() / cm.sum(1).clip(min=1), cm

f_rec, f_cm = recall_per_class(fp, ty)
x = np.arange(9)
fig, ax = plt.subplots(figsize=(11, 3.8))
if have_base:
    b_rec, b_cm = recall_per_class(bp, ty)
    ax.bar(x - 0.2, b_rec, 0.4, label="baseline 9-way", color="#9aa5b1")
ax.bar(x + (0.2 if have_base else 0), f_rec, 0.4, label="factorized", color="#2f6fd6")
ax.set_xticks(x); ax.set_xticklabels(NAMES, rotation=30); ax.set_ylim(0, 1)
ax.set_ylabel("test recall"); ax.set_title("Recall per action: baseline vs factorized"); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(CKPT_DIR, "recall_comparison.png"), dpi=150); plt.show()

print(f"{'action':10s} {'baseline':>9s} {'factorized':>11s} {'change':>8s}")
for i in range(9):
    b = b_rec[i] if have_base else float("nan")
    print(f"{NAMES[i]:10s} {b:9.3f} {f_rec[i]:11.3f} {f_rec[i] - b:+8.3f}")

## 8. The diagonal breakdown

For each diagonal: how often each model gets it **exactly**, gets **one of its two halves**, or gets **something else**. That's the table from your baseline analysis, now for both models.

In [ ]:
DIAGS = [i for i in range(9) if i % 3 != 1 and i // 3 != 1]          # UPLEFT, UPRIGHT, DOWNLEFT, DOWNRIGHT

def diag_breakdown(p, y):
    out = {}
    for d in DIAGS:
        m = y == d
        pd = p[m]
        exact = (pd == d).mean()
        half = (((pd % 3) == (d % 3)) ^ ((pd // 3) == (d // 3))).mean()    # exactly one axis right
        out[NAMES[d]] = (exact, half, 1 - exact - half)
    return out

print(f"{'':10s} {'exact':>15s} {'one half':>15s} {'other':>15s}")
print(f"{'':10s} {'base / fact':>15s} {'base / fact':>15s} {'base / fact':>15s}")
fb = diag_breakdown(fp, ty)
bb = diag_breakdown(bp, ty) if have_base else {k: (np.nan,) * 3 for k in fb}
for k in fb:
    print(f"{k:10s}" + "".join(f"{100 * bb[k][j]:7.0f} / {100 * fb[k][j]:<5.0f}" for j in range(3)))

## 9. Factorized confusion matrix

In [ ]:
cm_pct = f_cm / f_cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(9)); ax.set_xticklabels(NAMES, rotation=45, ha="right")
ax.set_yticks(range(9)); ax.set_yticklabels(NAMES)
for i in range(9):
    for j in range(9):
        ax.text(j, i, f"{100 * cm_pct[i, j]:.0f}", ha="center", va="center", fontsize=8,
                color="white" if cm_pct[i, j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Factorized model: test confusion matrix (%)")
plt.colorbar(im, fraction=0.046); plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "confusion_test.png"), dpi=150); plt.show()

## 10. Learning curves vs the baseline

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
base_hist_path = os.path.join(BASELINE_DIR, "results.json")
if os.path.exists(base_hist_path):
    bh = json.load(open(base_hist_path))["history"]
    ax.plot([h["epoch"] for h in bh], [h["macro_f1"] for h in bh], "o-", color="#9aa5b1", label="baseline 9-way")
ax.plot([h["epoch"] for h in history], [h["macro_f1"] for h in history], "o-", color="#2f6fd6", label="factorized")
ax.set_xlabel("epoch"); ax.set_ylabel("val macro-F1"); ax.set_ylim(0, 1); ax.legend()
ax.set_title("Validation macro-F1 per epoch")
plt.tight_layout(); plt.savefig(os.path.join(CKPT_DIR, "f1_curves.png"), dpi=150); plt.show()

## 11. Save

In [ ]:
results = {"run": RUN_NAME, "quick": QUICK, "seed": SEED, "epochs": EPOCHS, "lr": LR, "batch": BATCH,
           "params": count_params(model), "best_val_macro_f1": best_f1, "test": fact,
           "baseline_test": base if have_base else None, "history": history,
           "per_class_recall": {NAMES[i]: float(f_rec[i]) for i in range(9)},
           "diagonals": {k: dict(zip(["exact", "one_half", "other"], map(float, v))) for k, v in fb.items()}}
json.dump(results, open(os.path.join(CKPT_DIR, "results.json"), "w"), indent=2)
print("saved to", CKPT_DIR)

## Done

**QUICK run:** just check every cell runs. Then set `QUICK = False` → **Runtime → Run all** (~2 hours).

**After the real run, send me** the table from step 6, the recall table from step 7 and the diagonal breakdown from step 8.

**About one seed:** a difference of 0.01–0.02 macro-F1 between two single runs can be luck. If the result is close, the next step is to run both models with 2 more seeds (change `SEED`) and compare averages. If it's a big gap on the diagonals, one run is already convincing.